# [267] Virtual team 404

**Full names:** Marius Aleksander Brun, Thomas Mogstad Barth, Amund Dahlmo Berge  
**Student IDs (in order):** 109518, 172010, 109546  
**Kaggle team name:** [267] Virtual team 404

## Exploratory feature analysis


## Feature engineering



## Models we tried

### Decision tree

We implemented a decision tree classifier as a simple baseline for predicting unit commitment decisions. The model uses the same engineered features as the neural network, including hourly electricity prices, weekly inflow summaries, operational constraints, reservoir volumes, water values, and calendar information. This allows us to compare different model architectures while keeping the input features consistent.

We used a time-based train-validation split, training on cases before 2022 and validating on cases from 2022. A seven-day gap was introduced between the periods to prevent overlapping weekly input windows. Unlike the neural network, the decision tree does not require feature standardization, since its splits are based on feature thresholds rather than distances or gradient-based optimization.

To limit model complexity and reduce overfitting, we restricted the tree to a maximum depth of 5 and required at least 20 samples per leaf.

The 2022 validation micro ROC-AUC we achieved with this model was quite low at 0.6736.

In [ ]:
tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
tree.fit(X_train, y_train)

### Simple neural network

We experimented with multilayer perceptron (MLP) classifiers with different hidden-layer architectures to investigate how network capacity affected predictive performance. The input features were the same as in the decision tree above.

Before training, we standardized the input features using `StandardScaler`, fitted exclusively on the training data to prevent data leakage. We used a time-based split, training on cases before 2022 and validating on cases from 2022, with a seven-day gap between the periods to stop overlapping weekly input windows.

We compared five network architectures using the same feature set and validation procedure:

| Hidden-layer architecture | Validation micro ROC-AUC |
|---|---:|
| (128,) | 0.9134 |
| (1024, 512, 256, 128) | 0.9161 |
| (512, 256) | 0.9185 |
| (512, 256, 128) | 0.9211 |
| (256, 128) | **0.9240** |

The network with hidden layers of 256 and 128 neurons achieved the best validation score of 0.9240. Increasing the network size did not improve performance.

We also experimented with different learning rates. Increasing the learning rate resulted in a significant performance drop. We therefore retained a learning rate of 0.001 for the best-performing configuration.

### Model interpretation of simple neural network

We compared three input groups by shuffling each group between 2022 validation
cases and measuring the decrease in micro ROC-AUC. The model stayed the same.
A larger decrease means greater reliance on that group. Columns in each
group were shuffled together to preserve their within-week relationships.
We did one shuffle per group, so the result does not
show causality or the direction of an effect. and got the following values:

Initial water values        0.018562
Hourly prices               0.172482
End-of-week water values    0.368787

This tells us that the model relies most heavily on the end-of-week water values.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
mlp = MLPClassifier((256, 128), max_iter=2000, early_stopping=True, learning_rate_init=0.001, random_state=67)
mlp.fit(X_train_scaled, y_train)